In [7]:
!pip install -q transformers datasets evaluate jiwer pytorch-lightning

In [8]:
import os
import json
import torch
import glob
import evaluate
from datasets import Dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import pytorch_lightning as pl
from torch.utils.data import DataLoader

# ==========================================
# 0. НАЛАШТУВАННЯ
# ==========================================
DATASET_SIZE_PERCENT = 0.5  # Кількість даних
model_id = "KSE-RESEARCH-Group/whisper-small-dido-yvanchyk-v2" 

torch.cuda.empty_cache()

# ==========================================
# 1. ПІДГОТОВКА ДАНИХ ТА ПАРСИНГ JSON
# ==========================================
base_path = "/kaggle/input/datasets/dimonlg/toronto-lab"
labels_path = os.path.join(base_path, "labels.jsonl")

print(f"Зчитування labels.jsonl для моделі: {model_id}")
with open(labels_path, 'r', encoding='utf-8') as f:
    try:
        data = json.load(f)
        labels_dict = data.get("root", data)
    except json.JSONDecodeError:
        labels_dict = {}
        f.seek(0)
        for line in f:
            if line.strip(): labels_dict.update(json.loads(line))

clean_labels = {k.replace("dataset/", ""): v for k, v in labels_dict.items()}
audio_files = glob.glob(os.path.join(base_path, "toronto_*/*.wav"))

data_list = []
test_lines = ['toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89', 'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7', 'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81', 'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166', 'toronto_58']

for file_path in audio_files:
    rel_path = "/".join(file_path.split("/")[-2:])
    if rel_path in clean_labels:
        data_list.append({"audio": file_path, "text": clean_labels[rel_path], "file_id": os.path.basename(file_path).split('.')[0]})

full_dataset = Dataset.from_list(data_list)
test_dataset = full_dataset.filter(lambda x: any(x["file_id"] == t or x["file_id"].startswith(t + "_") for t in test_lines))
train_val_dataset = full_dataset.filter(lambda x: not any(x["file_id"] == t or x["file_id"].startswith(t + "_") for t in test_lines))

if DATASET_SIZE_PERCENT < 1.0:
    train_val_dataset = train_val_dataset.shuffle(seed=42).select(range(int(len(train_val_dataset) * DATASET_SIZE_PERCENT)))
    test_dataset = test_dataset.shuffle(seed=42).select(range(int(len(test_dataset) * DATASET_SIZE_PERCENT)))

train_val_split = train_val_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_val_split["train"].cast_column("audio", Audio(sampling_rate=16000))
val_dataset = train_val_split["test"].cast_column("audio", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

# ==========================================
# 2. ПРОЦЕСОР ТА ОБРОБКА
# ==========================================
processor = WhisperProcessor.from_pretrained(model_id, language="ukrainian", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

print("Обробка ознак...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names)
test_dataset = test_dataset.map(prepare_dataset, remove_columns=test_dataset.column_names)

class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor): self.processor = processor
    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        batch["labels"] = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        return batch

collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collator, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collator, num_workers=2)

# ==========================================
# 3. МОДЕЛЬ ТА ТРЕНУВАННЯ
# ==========================================
class WhisperASR(pl.LightningModule):
    def __init__(self, model_id, processor):
        super().__init__()
        self.model = WhisperForConditionalGeneration.from_pretrained(model_id)
        self.processor = processor
        self.wer_metric = evaluate.load("wer")
        self.cer_metric = evaluate.load("cer")
        
        forced_decoder_ids = self.processor.get_decoder_prompt_ids(language="ukrainian", task="transcribe")
        self.model.generation_config.forced_decoder_ids = forced_decoder_ids
        self.model.generation_config.suppress_tokens = []
        
        if hasattr(self.model.generation_config, "language"):
            self.model.generation_config.language = None

    def forward(self, input_features, labels=None): 
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        loss = self(batch["input_features"], batch["labels"]).loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def _eval_step(self, batch, prefix):
        input_features, labels = batch["input_features"], batch["labels"]
        predicted_ids = self.model.generate(input_features)
        
        labels[labels == -100] = self.processor.tokenizer.pad_token_id
        pred_str = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)
        label_str = self.processor.batch_decode(labels, skip_special_tokens=True)
        
        wer = self.wer_metric.compute(predictions=pred_str, references=label_str)
        cer = self.cer_metric.compute(predictions=pred_str, references=label_str)
        
        self.log(f"{prefix}_wer", wer, on_epoch=True, prog_bar=True)
        self.log(f"{prefix}_cer", cer, on_epoch=True, prog_bar=True)
        return {"wer": wer}

    def validation_step(self, batch, batch_idx): return self._eval_step(batch, "val")
    def test_step(self, batch, batch_idx): return self._eval_step(batch, "test")

    def configure_optimizers(self): 
        return torch.optim.AdamW(self.parameters(), lr=1e-5)

# ==========================================
# 4. ЗАПУСК
# ==========================================
model = WhisperASR(model_id, processor)

trainer = pl.Trainer(
    max_epochs=2, # 2 епохи
    accelerator="gpu", 
    devices=1, 
    precision="16-mixed",
    log_every_n_steps=50
)

print(f"\nСтарт навчання {model_id}...")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print("\nФінальне тестування...")
trainer.test(model, dataloaders=test_loader)

Зчитування labels.jsonl для моделі: KSE-RESEARCH-Group/whisper-small-dido-yvanchyk-v2


Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Обробка ознак...


Map:   0%|          | 0/5742 [00:00<?, ? examples/s]

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Фінальне тестування...


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_cer          │    0.2464270144701004     │
│         test_wer          │    0.42678606510162354    │
└───────────────────────────┴───────────────────────────┘

[{'test_wer': 0.42678606510162354, 'test_cer': 0.2464270144701004}]

In [9]:
import IPython.display as ipd
import random

def visualize_results(model, dataset, processor, num_samples=5):
    model.eval()
    # Випадкові індекси
    indices = random.sample(range(len(dataset)), num_samples)
    
    print(f"🔍 Демонстрація роботи моделі на {num_samples} випадкових прикладах з TEST SET:\n")
    print("-" * 80)

    for i, idx in enumerate(indices):
        item = dataset[idx]
        input_features = torch.tensor([item["input_features"]]).to(model.device)
        
        # Генерація тексту
        with torch.no_grad():
            predicted_ids = model.model.generate(input_features)
        
        prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        reference = processor.tokenizer.decode(item["labels"], skip_special_tokens=True)
        
        # Виведення результатів
        print(f"Приклад №{i+1}")
        print(f"📝 ОРИГІНАЛ: {reference}")
        print(f"🤖 МОДЕЛЬ  : {prediction}")
        
        print("-" * 80)

# Запуск візуалізації
visualize_results(model, test_dataset, processor)

🔍 Демонстрація роботи моделі на 5 випадкових прикладах з TEST SET:

--------------------------------------------------------------------------------
Приклад №1
📝 ОРИГІНАЛ: Хм-хм.  Мова – душа народу. Вперед, до копу!
🤖 МОДЕЛЬ  : Мова душа народу. Вперед до копу.
--------------------------------------------------------------------------------
Приклад №2
📝 ОРИГІНАЛ: А там, знаєш,… Ти, м... мабуть, ми з Павлом там, де рєльси. Якщо трамваєв ніде немає. Небезпечно, але
🤖 МОДЕЛЬ  : А там, знаєш, ми б мабуть там спевлом дирілці. Якщо трамваївні де немає, небезпечно. Але
--------------------------------------------------------------------------------
Приклад №3
📝 ОРИГІНАЛ: попобності представникам російської влади, умисних діях, учинених з метою зміни державного кордону
🤖 МОДЕЛЬ  : пособності представникам російської влади у мисних діях, очиненах з матою, зміниваю державного кордону,
--------------------------------------------------------------------------------
Приклад №4
📝 ОРИГІНАЛ: Так, зв